# 07 - Level 0 Baseline

Reads `data/processed/panel_train.parquet` / `panel_test.parquet` and the accompanying
`model_table_schema.json`, built by `scripts/01_load.py`, `02_clean_validate.py` and
`03_build_panel.py`. Read-only; the split, the features and the leakage proofs are owned by
stage 3.

Builds the Level 0 ladder: three naive forecasts, a per-household degree-day regression, and
one pooled Random Forest predicting `gross_load` one day ahead.

Writes `reports/level0_baseline_report.md`, `reports/level0_energy_signatures.csv`,
`reports/level0_per_household_errors.csv` and `data/processed/level0_test_predictions.parquet`.

Note on the weather. Stage 3 shifts every weather column by one day, so the panel carries the
d-1 observation, not the target-day value (`weather_shift_days = 1`). The forecast is therefore
operational rather than ex-post, and accuracy is not inflated by weather the model would not
have on the day.

Metrics: MAE (primary), RMSE, MASE. No R2.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

_here = Path.cwd()
REPO_ROOT = next((p for p in [_here, *_here.parents] if (p / "data" / "processed").is_dir()), _here)
PROCESSED = REPO_ROOT / "data" / "processed"
REPORTS_DIR = REPO_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

plt.rcParams.update({"figure.figsize": (13, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
sns.set_palette("tab10")
print(f"REPO_ROOT : {REPO_ROOT}")

In [ ]:
# Read the stage-3 output plus its schema. The schema is the contract: feature groups, target,
# split cutoff and the weather shift all come from there rather than being retyped here.
train = pd.read_parquet(PROCESSED / "panel_train.parquet")
test = pd.read_parquet(PROCESSED / "panel_test.parquet")
SCHEMA = json.loads((PROCESSED / "model_table_schema.json").read_text())

TARGET = SCHEMA["target"]
SPLIT_CUTOFF = pd.Timestamp(SCHEMA["split_cutoff"])
WEATHER_SHIFT = SCHEMA["weather_shift_days"]
GROUPS = SCHEMA["feature_groups"]
SCHEMA_FEATURES = [c for g in GROUPS.values() for c in g]

print(f"train : {len(train):,} rows x {train.shape[1]} cols, "
      f"{train['Household_ID'].nunique()} households, {train['date'].min().date()} -> {train['date'].max().date()}")
print(f"test  : {len(test):,} rows x {test.shape[1]} cols, "
      f"{test['Household_ID'].nunique()} households, {test['date'].min().date()} -> {test['date'].max().date()}")
print(f"target={TARGET} | split cutoff={SPLIT_CUTOFF.date()} | weather shift={WEATHER_SHIFT} day(s)")
print(f"schema hands over {len(SCHEMA_FEATURES)} active features in {len(GROUPS)} groups")

In [ ]:
# Fail-loud contract gate, driven by the schema rather than a hardcoded column list.
missing = [c for c in SCHEMA_FEATURES + [TARGET, "Household_ID", "date"] if c not in train.columns]
assert not missing, f"columns promised by the schema but absent from the panel: {missing}"
assert set(train.columns) == set(test.columns), "train/test column-set drift"
assert train[TARGET].notna().all() and test[TARGET].notna().all(), "target must be present on every row"
assert train["date"].max() <= test["date"].min(), "train/test temporal order violated"
assert WEATHER_SHIFT >= 1, "weather must be lagged; a shift of 0 would be a perfect-forecast assumption"
print("Contract OK: schema columns present, no target NaN, temporal order holds, weather lagged.")

# NaNs are retained by stage 3 on purpose (is_modelable = target present, nothing else).
_nan = train[SCHEMA_FEATURES].isna().sum()
print("\nFeature NaN counts in train (0 not shown):")
print(_nan[_nan > 0].to_string() if (_nan > 0).any() else "  none")

In [ ]:
def df_to_md(df):
    """Render a DataFrame as a GitHub-flavoured markdown table (no external lib)."""
    cols = list(df.columns)
    if df.shape[0] == 0:
        return "_(no rows)_"
    lines = ["| " + " | ".join(map(str, cols)) + " |",
             "| " + " | ".join("---" for _ in cols) + " |"]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(str(v) for v in row.values) + " |")
    return "\n".join(lines)

## STEP 0 - Data-understanding findings, rechecked on the train panel

Three findings drive decisions here. The EDA ran on the raw cohort and this panel is a filtered
subset, so recompute them before relying on them.

1. Portfolio load tracks temperature. -> keep `hdd_15`, build the energy-signature benchmark
   (STEP 4b), check residuals against temperature (STEP 7c).
2. Per-household consumption is right-skewed. -> household encoding, per-household error
   distribution (STEP 7b), MASE scaled per household.
3. PV households draw less on sunny days and are more volatile, but the groups overlap.
   -> keep `Installation_HasPVSystem`. The overlap is what Level 1 targets.

Because the weather is now the d-1 observation, these correlations are measured against lagged
weather and are expected to be weaker than the same-day figures reported by the EDA.

In [ ]:
# STEP 0 - RECHECK THE DATA-UNDERSTANDING FINDINGS ON THE TRAIN PANEL
# Train only: these are modelling assumptions, so they must hold without touching test.
daily = train.groupby("date").agg(load_median=(TARGET, "median"),
                                  temp_median=("Temperature_avg_hourly_mean", "median"),
                                  hdd_median=("hdd_15", "median"))
R_TEMP = daily["load_median"].corr(daily["temp_median"])
R_HDD = daily["load_median"].corr(daily["hdd_median"])
print(f"(1) corr(portfolio median load, median temperature d-1) r = {R_TEMP:+.3f}")
print(f"    corr(portfolio median load, median hdd_15 d-1)      r = {R_HDD:+.3f}")

hh_median_load = train.groupby("Household_ID", observed=True)[TARGET].median()
SPREAD_RATIO = hh_median_load.max() / hh_median_load.min()
print(f"(2) per-household median daily load: min {hh_median_load.min():.2f}, "
      f"p10 {hh_median_load.quantile(.10):.2f}, median {hh_median_load.median():.2f}, "
      f"p90 {hh_median_load.quantile(.90):.2f}, max {hh_median_load.max():.2f} kWh "
      f"({hh_median_load.shape[0]} train households)")
print(f"    max/min = {SPREAD_RATIO:.0f}x, but both extremes are short-history households; "
      f"p90/p10 = {hh_median_load.quantile(.90)/hh_median_load.quantile(.10):.1f}x is the robust view")

hh_pv = (train.groupby("Household_ID", observed=True)["Installation_HasPVSystem"].first().astype(str) == "True")
hh_cv = (train.groupby("Household_ID", observed=True)[TARGET].std()
         / train.groupby("Household_ID", observed=True)[TARGET].mean())
PV_TRUE, PV_FALSE = int(hh_pv.sum()), int((~hh_pv).sum())
CV_PV, CV_NONPV = hh_cv[hh_pv].median(), hh_cv[~hh_pv].median()
print(f"(3) {PV_TRUE} PV vs {PV_FALSE} non-PV households in train")
print(f"    median per-household CV of daily load: PV {CV_PV:.2f} vs non-PV {CV_NONPV:.2f}")

## STEP 0b - The same check at household resolution

Finding 1 is about the portfolio. The energy signature in STEP 4b assumes the relationship also
holds inside a single household, which is a different claim. Measure it directly.

Train only.

In [ ]:
# STEP 0b - HOUSEHOLD-LEVEL CORRELATION BETWEEN gross_load AND hdd_15
# Vectorised Pearson per household (no groupby.apply, so it does not depend on the pandas version).
_g = train[["Household_ID", TARGET, "hdd_15"]].dropna()
_mean = _g.groupby("Household_ID", observed=True)[[TARGET, "hdd_15"]].transform("mean")
_dev = _g[[TARGET, "hdd_15"]] - _mean
_key = _g["Household_ID"]
_num = (_dev[TARGET] * _dev["hdd_15"]).groupby(_key, observed=True).sum()
_den = np.sqrt((_dev[TARGET] ** 2).groupby(_key, observed=True).sum()
               * (_dev["hdd_15"] ** 2).groupby(_key, observed=True).sum())
_n_days = _g.groupby("Household_ID", observed=True).size()

# Same 30-day threshold STEP 4b uses to decide which households get their own signature.
hh_r_hdd = (_num / _den.replace(0, np.nan))[_n_days >= 30].dropna()

HH_R_MEDIAN = hh_r_hdd.median()
HH_R_P25, HH_R_P75 = hh_r_hdd.quantile(0.25), hh_r_hdd.quantile(0.75)
HH_R_MIN, HH_R_MAX = hh_r_hdd.min(), hh_r_hdd.max()
HH_R2_MEDIAN = float((hh_r_hdd ** 2).median())
HH_N_CORR = int(len(hh_r_hdd))

print("Correlation with hdd_15 at two levels of aggregation")
print("-" * 62)
print(f"  portfolio (daily medians)      r = {R_HDD:+.3f}   ->  R2 = {R_HDD ** 2:.3f}")
print(f"  household-day, median of {HH_N_CORR:>3} hh  r = {HH_R_MEDIAN:+.3f}   ->  R2 = {HH_R2_MEDIAN:.3f}")
print(f"  household-day, IQR             r = {HH_R_P25:+.3f} to {HH_R_P75:+.3f}")
print(f"  household-day, full range      r = {HH_R_MIN:+.3f} to {HH_R_MAX:+.3f}")
print()
print(f"Explained variance: {R_HDD ** 2:.0%} at portfolio level -> median {HH_R2_MEDIAN:.0%} per household.")
print("STEP 4b tests whether what is left is enough for a temperature-only forecast.")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(hh_r_hdd, bins=25, edgecolor="white")
ax.axvline(R_HDD, color="crimson", lw=2, label=f"portfolio r = {R_HDD:+.2f}")
ax.axvline(HH_R_MEDIAN, color="black", lw=2, ls="--", label=f"household median r = {HH_R_MEDIAN:+.2f}")
ax.set_xlabel("corr(gross_load, hdd_15) within a household")
ax.set_ylabel("households")
ax.set_title("Household-level correlation vs the portfolio value")
ax.legend()
plt.tight_layout(); plt.show()

## STEP 0c - How far does temperature alone reach?

Stage 3 made two decisions this step tests. It derived `hdd_15` as the heat-pump-specific
temperature channel, and it dated every weather column to d-1 because that is what exists when
the bid is placed. The first is a hypothesis about the physics, the second is a constraint on
what the model may use.

Separating them needs the same-day observation, which is deliberately absent from the modelling
panel. It is read here from `data/interim/02_clean/weather_daily.parquet` for a descriptive
statistic only. Nothing from this cell enters a model.

In [ ]:
# STEP 0c - ENERGY SIGNATURE AS A CHARACTERISATION (same-day vs lagged temperature)
# Descriptive only: quantifies how much of a household's daily consumption temperature explains,
# and how much of that survives the one-day lag the forecasting task imposes.
_wd = pd.read_parquet(REPO_ROOT / "data" / "interim" / "02_clean" / "weather_daily.parquet")
_wd["date"] = pd.to_datetime(_wd["date"])
HDD_BASE = SCHEMA["hdd_base_c"]
_wd["hdd_same"] = (HDD_BASE - _wd["Temperature_avg_hourly_mean"]).clip(lower=0)
_wd = _wd.rename(columns={"Weather_ID": "weather_id"})[["weather_id", "date", "hdd_same"]]
_wd["weather_id"] = _wd["weather_id"].astype(str)

def _attach_same_day(df):
    d = df.copy(); d["weather_id"] = d["weather_id"].astype(str)
    return d.merge(_wd, on=["weather_id", "date"], how="left")

train_c, test_c = _attach_same_day(train), _attach_same_day(test)

def _per_hh_r(df, col):
    _g = df[["Household_ID", TARGET, col]].dropna()
    m = _g.groupby("Household_ID", observed=True)[[TARGET, col]].transform("mean")
    dv = _g[[TARGET, col]] - m
    k = _g["Household_ID"]
    num = (dv[TARGET] * dv[col]).groupby(k, observed=True).sum()
    den = np.sqrt((dv[TARGET] ** 2).groupby(k, observed=True).sum()
                  * (dv[col] ** 2).groupby(k, observed=True).sum())
    n = _g.groupby("Household_ID", observed=True).size()
    return (num / den.replace(0, np.nan))[n >= MIN_TRAIN_DAYS].dropna()

r_same = _per_hh_r(train_c, "hdd_same")
r_lag = _per_hh_r(train_c, "hdd_15")
R2_SAME, R2_LAG = float((r_same ** 2).median()), float((r_lag ** 2).median())
R_SAME_MED, R_LAG_MED = float(r_same.median()), float(r_lag.median())

print("Within-household correlation between daily gross load and heating degree days")
print("-" * 74)
print(f"  same-day temperature (the physical relationship) : median r = {R_SAME_MED:+.3f}  ->  R2 = {R2_SAME:.3f}")
print(f"  temperature from d-1 (what the model may use)    : median r = {R_LAG_MED:+.3f}  ->  R2 = {R2_LAG:.3f}")
print(f"\n  {R2_SAME:.0%} of within-household variance is temperature-driven; {R2_LAG:.0%} survives the one-day lag.")
print(f"  The {R2_SAME - R2_LAG:.0%} difference is what a weather forecast, rather than yesterday's reading,")
print("  could in principle recover.")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(r_same, bins=25, alpha=0.65, label=f"same-day (median {R_SAME_MED:+.2f})")
ax.hist(r_lag, bins=25, alpha=0.65, label=f"d-1 (median {R_LAG_MED:+.2f})")
ax.set_xlabel("corr(gross_load, heating degree days) within a household")
ax.set_ylabel("households")
ax.set_title("Temperature dependence, before and after the one-day lag")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# STEP 1 - HOUSEHOLD IDENTITY ENCODING
# The schema's static group has no household identity. We add one at the modelling stage: a
# pooled model needs a numeric handle on "which household is this" to learn per-household level
# offsets. Households seen only in test get an explicit unseen code, so no fitted label leaks.
hh_codes = {hh: i for i, hh in enumerate(sorted(train["Household_ID"].unique()))}
UNSEEN = -1
for _d in (train, test):
    _d["household_enc"] = _d["Household_ID"].map(hh_codes).fillna(UNSEEN).astype(int)

# Sunshine is absent at three of the eight stations, and the gap is structural rather than
# random: a station either reports it on every day or on none. That is information, so it gets
# an explicit indicator instead of being imputed away.
for _d in (train, test):
    _d["sunshine_missing"] = _d["Sunshine_duration_hourly_sum"].isna().astype(int)

N_UNSEEN_TEST = int((test["household_enc"] == UNSEEN).sum())
print(f"Households encoded from train : {len(hh_codes)}")
print(f"Test rows with unseen household ({UNSEEN}) : {N_UNSEEN_TEST} ({N_UNSEEN_TEST/len(test):.1%})")
print(f"Sunshine missing: train {train['sunshine_missing'].mean():.1%}, test {test['sunshine_missing'].mean():.1%} of rows")
print("  by station (train):",
      train.groupby(train["weather_id"].astype(str), observed=True)["sunshine_missing"].mean().round(2).to_dict())

## Feature selection - method

Stage 3 hands over an active feature set and keeps a documented reserve. That set is the
candidate pool here, not the final answer: the pipeline deliberately keeps variables a
tree-based model can tolerate, and leaves modelling-stage pruning to the modelling stage.

Three steps, unchanged from the previous baseline:

1. Group candidates by role.
2. Within each group drop near-duplicates from the train correlation matrix, keeping the most
   domain-meaningful representative. Numbers in STEP 2.
3. Confirm the hyperparameter choice on a time-based validation slice, STEP 5.

Added to the pool beyond the schema: `household_enc` and `sunshine_missing` from STEP 1.

In [ ]:
# STEP 2 - CANDIDATE FEATURE AUDIT (collinearity)
CANDIDATES = SCHEMA_FEATURES + ["household_enc", "sunshine_missing"]
_num = [c for c in CANDIDATES if pd.api.types.is_numeric_dtype(train[c])]
corr = train[_num + [TARGET]].corr()

TEMP_FAMILY = ["Temperature_avg_hourly_mean", "Temperature_avg_hourly_min",
               "Temperature_avg_hourly_max", "DewPoint_hourly_mean", "hdd_15"]
print("Temperature family |r| (train):")
display(corr.loc[TEMP_FAMILY, TEMP_FAMILY].abs().round(3))
_m = corr.loc[TEMP_FAMILY, TEMP_FAMILY].abs().where(~np.eye(len(TEMP_FAMILY), dtype=bool))
print(f"  weakest pair |r| = {_m.min().min():.3f} ; hdd_15 against the four raw channels: "
      f"{corr.loc['hdd_15', [c for c in TEMP_FAMILY if c != 'hdd_15']].abs().min():.3f} to "
      f"{corr.loc['hdd_15', [c for c in TEMP_FAMILY if c != 'hdd_15']].abs().max():.3f}")

print("\nCorrelation with the target:")
for c in TEMP_FAMILY + ["Sunshine_duration_hourly_sum", "Humidity_avg_hourly_mean",
                        "WindSpeed_hourly_mean", "Precipitation_total_hourly_sum"]:
    print(f"  {c:34} r = {corr.loc[c, TARGET]:+.3f}")

print("\nSecondary weather against hdd_15:")
for c in ["Sunshine_duration_hourly_sum", "Humidity_avg_hourly_mean",
          "WindSpeed_hourly_mean", "Precipitation_total_hourly_sum"]:
    print(f"  {c:34} |r| = {abs(corr.loc[c, 'hdd_15']):.3f}")

AR = ["recv_lag1", "recv_lag7", "recv_roll7_mean", "recv_roll7_std"]
_ca = corr.loc[AR, AR].abs()
_mm = _ca.where(~np.eye(4, dtype=bool))
print(f"\nAutoregressive family: max pairwise |r| = {np.nanmax(_mm.values):.3f}")

# determinism checks behind the two calendar drops
_ct = pd.crosstab(train["dow"].astype(str), train["is_weekend"].astype(str))
_ct2 = pd.crosstab(train["month"].astype(str), train["season"].astype(str))
print(f"is_weekend an exact function of dow: {(_ct > 0).sum(axis=1).le(1).all()}")
print(f"season an exact coarsening of month: {(_ct2 > 0).sum(axis=1).le(1).all()}")

## Feature selection - decisions

| Group | Candidates | Decision | Why |
|---|---|---|---|
| Temperature | `Temperature_avg_hourly_mean/min/max`, `DewPoint_hourly_mean`, `hdd_15` | **Keep `hdd_15` only** | The five carry one signal; `hdd_15` correlates with each raw channel at high \|r\| (numbers in STEP 2). It is the domain-correct, heat-pump-specific transform, clipped at the 15C comfort threshold. |
| Solar | `Sunshine_duration_hourly_sum`, `sunshine_missing` | **Keep both** | Daily sunshine duration is the only direct observation of the driver behind PV self-consumption. Its absence is structural at three of eight stations, so the indicator carries real information rather than papering over a gap. |
| Secondary weather | `Humidity_avg_hourly_mean`, `WindSpeed_hourly_mean`, `Precipitation_total_hourly_sum` | **Keep all three** | Weakly correlated with `hdd_15` and physically distinct (moisture, ventilation loss, precipitation). |
| Calendar (weekly) | `dow`, `is_weekend` | **Keep `dow`, drop `is_weekend`** | `is_weekend` is an exact function of `dow`. The tree can rediscover the weekend split from the finer variable. |
| Calendar (yearly) | `month`, `season` | **Keep `month`, drop `season`** | `season` is an exact 4-bucket coarsening of `month`. |
| Calendar (other) | `is_holiday` | **Keep** | Orthogonal to `dow`/`month`. |
| Autoregressive | `recv_lag1`, `recv_lag7`, `recv_roll7_mean`, `recv_roll7_std` | **Keep all four** | Correlated but not redundant; each plays a distinct role, and the first three double as the naive baselines. |
| Household identity | `household_enc`, `Installation_HasPVSystem` | **Keep both** | The encoding gives the pooled model per-household level offsets; the PV flag gives it the group difference. |
| Station identity | `weather_id` | **Drop** | Its predictive content is the weather it stands for, which is already in the set explicitly. Keeping it lets the forest key on station as a proxy for household group. |
| Regime | `is_post_visit` | **Keep** | A dated regime shift no other feature encodes. |

In [ ]:
# STEP 3 - FREEZE FEATURE SET + BUILD MODEL MATRICES
DROPPED = {"Temperature_avg_hourly_mean", "Temperature_avg_hourly_min", "Temperature_avg_hourly_max",
           "DewPoint_hourly_mean", "is_weekend", "season", "weather_id"}
FEATURE_COLS = [c for c in CANDIDATES if c not in DROPPED]

X_train, y_train = train[FEATURE_COLS].copy(), train[TARGET]
X_test, y_test = test[FEATURE_COLS].copy(), test[TARGET]

# Trees split on raw values, so low-cardinality categorical/boolean columns just need an
# integer code. Nullable booleans become float so that a genuine <NA> stays NaN.
for col in FEATURE_COLS:
    if str(X_train[col].dtype) in ("category", "bool", "boolean"):
        X_train[col] = X_train[col].astype("float64") if str(X_train[col].dtype) == "boolean" \
            else X_train[col].astype("category").cat.codes
        X_test[col] = X_test[col].astype("float64") if str(X_test[col].dtype) == "boolean" \
            else X_test[col].astype("category").cat.codes

print(f"Candidates {len(CANDIDATES)} -> dropped {len(DROPPED)} -> final feature count {len(FEATURE_COLS)}")
print(f"X_train {X_train.shape}   X_test {X_test.shape}")
print("Features:", ", ".join(FEATURE_COLS))

# Evaluation mask. The three naive forecasts need their lag columns, so every model is scored on
# the rows where all of them can produce a prediction. Scoring models on different row sets
# would make the comparison meaningless.
EVAL_COLS = ["recv_lag1", "recv_lag7", "recv_roll7_mean"]
eval_train = train[EVAL_COLS].notna().all(axis=1)
eval_test = test[EVAL_COLS].notna().all(axis=1)
N_EVAL_TEST = int(eval_test.sum())
print(f"\nEvaluation rows: train {int(eval_train.sum()):,}/{len(train):,} ({eval_train.mean():.1%}), "
      f"test {N_EVAL_TEST:,}/{len(test):,} ({eval_test.mean():.1%})")
print("The forest still trains on every train row; only scoring is restricted to the shared subset.")

## STEP 4 - Naive baselines and metric definitions

Three naive forecasts, all reusing leakage-safe lag columns built by stage 3:

- lag-1: tomorrow = today (`recv_lag1`)
- lag-7: tomorrow = same weekday last week (`recv_lag7`)
- 7-day rolling mean: tomorrow = trailing 7-day average (`recv_roll7_mean`)

Metrics:

- **MAE** (primary), kWh.
- **RMSE**, for the tail days.
- **MASE**: absolute test error divided by that household's train-period MAE under naive lag-1,
  then averaged. Below 1 beats persistence. Households unseen in train, and degenerate zero
  denominators, fall back to the fleet-median scale; both counts are printed.

In [ ]:
# STEP 4 - METRIC DEFINITIONS + NAIVE BASELINES
_ins = (train.loc[eval_train, TARGET] - train.loc[eval_train, "recv_lag1"]).abs()
Q_h = _ins.groupby(train.loc[eval_train, "Household_ID"], observed=True).mean()
Q_h = Q_h.where(Q_h > 1e-9)
FLEET_Q = Q_h.median()

_scale = test.loc[eval_test, "Household_ID"].map(Q_h)
N_FALLBACK_ROWS = int(_scale.isna().sum())
mase_scale_test = _scale.fillna(FLEET_Q).values

y_eval = test.loc[eval_test, TARGET].values
print(f"MASE scale: per-household in-sample naive lag-1 MAE; fleet median = {FLEET_Q:.3f} kWh")
print(f"Test rows on the fleet-median fallback: {N_FALLBACK_ROWS} ({N_FALLBACK_ROWS/N_EVAL_TEST:.1%})")

def compute_metrics(y_pred, label):
    y_pred = np.asarray(y_pred, float)
    assert len(y_pred) == len(y_eval), "metrics are defined on the shared evaluation rows"
    abs_err = np.abs(y_eval - y_pred)
    mae = abs_err.mean()
    rmse = float(np.sqrt(np.mean((y_eval - y_pred) ** 2)))
    mase = float(np.mean(abs_err / mase_scale_test))
    print(f"  {label:<38}  MAE = {mae:6.3f} kWh  |  RMSE = {rmse:6.3f} kWh  |  MASE = {mase:.3f}")
    return {"Model": label, "MAE": mae, "RMSE": rmse, "MASE": mase}

results = []
print("\n" + "=" * 80)
print(f"Naive baselines (test set, {N_EVAL_TEST:,} shared evaluation rows)")
print("=" * 80)
results.append(compute_metrics(test.loc[eval_test, "recv_lag1"], "Naive lag-1 (yesterday)"))
results.append(compute_metrics(test.loc[eval_test, "recv_lag7"], "Naive lag-7 (same weekday)"))
results.append(compute_metrics(test.loc[eval_test, "recv_roll7_mean"], "Naive 7-day rolling mean"))

## STEP 4b - Energy-signature benchmark

Per household, OLS of daily `gross_load` on `hdd_15`. Intercept is the base load, slope the
heating response in kWh per degree-day.

Households with at least 30 train days and non-degenerate `hdd_15` get their own coefficients;
the rest use pooled coefficients from the full train panel. Predictions floored at zero.

No autoregressive memory, so the gap to naive lag-1 measures what persistence adds and the gap
to the Random Forest measures what the rest of the feature set adds.

In [ ]:
# STEP 4b - ENERGY-SIGNATURE BENCHMARK
MIN_TRAIN_DAYS = 30
sig_a, sig_b = {}, {}
for hh, g in train.groupby("Household_ID", observed=True):
    g = g[["hdd_15", TARGET]].dropna()
    if len(g) >= MIN_TRAIN_DAYS and g["hdd_15"].std() > 1e-6:
        slope, intercept = np.polyfit(g["hdd_15"].values, g[TARGET].values, 1)
        sig_a[hh], sig_b[hh] = intercept, slope

_tr = train[["hdd_15", TARGET]].dropna()
pooled_slope, pooled_intercept = np.polyfit(_tr["hdd_15"].values, _tr[TARGET].values, 1)

def signature_predict(df):
    a = df["Household_ID"].map(sig_a).fillna(pooled_intercept).values
    b = df["Household_ID"].map(sig_b).fillna(pooled_slope).values
    hdd = df["hdd_15"].fillna(df["hdd_15"].median()).values
    return np.clip(a + b * hdd, 0.0, None)

N_SIG_HH = len(sig_a)
_pooled_rows = int(test.loc[eval_test, "Household_ID"].map(sig_a).isna().sum())
print(f"Per-household signatures fitted : {N_SIG_HH} / {train['Household_ID'].nunique()} train households")
print(f"Test rows on the pooled fallback: {_pooled_rows} ({_pooled_rows/N_EVAL_TEST:.1%})")
print(f"Median fitted signature         : base load {np.median(list(sig_a.values())):.2f} kWh, "
      f"heating response {np.median(list(sig_b.values())):.2f} kWh per degree-day")
print(f"Pooled fallback signature       : base load {pooled_intercept:.2f} kWh, "
      f"heating response {pooled_slope:.2f} kWh per degree-day\n")

y_pred_sig = signature_predict(test.loc[eval_test])
results.append(compute_metrics(y_pred_sig, "Energy signature (HDD regression)"))

In [ ]:
# STEP 4c - SAVE THE FITTED SIGNATURES
# Base load and heating response per household are interpretable results in their own right,
# and Level 1 reuses the fitted prediction as an input feature.
sig_df = pd.DataFrame({"Household_ID": list(sig_a.keys()),
                       "base_load_kwh": [sig_a[h] for h in sig_a],
                       "heating_response_kwh_per_hdd": [sig_b[h] for h in sig_a]})
sig_df["train_days"] = sig_df["Household_ID"].map(train.groupby("Household_ID", observed=True).size())
sig_df["r_hdd_load"] = sig_df["Household_ID"].map(hh_r_hdd)
sig_df["r2_hdd_load"] = sig_df["r_hdd_load"] ** 2
sig_df = sig_df.sort_values("Household_ID").reset_index(drop=True)
SIG_PATH = REPORTS_DIR / "level0_energy_signatures.csv"
sig_df.to_csv(SIG_PATH, index=False)
print(f"Saved {len(sig_df)} fitted signatures -> {SIG_PATH.relative_to(REPO_ROOT)}")
display(sig_df.describe().round(2))

## STEP 5 - Hyperparameters

`n_estimators=300` and `max_features="sqrt"` stay at the regression defaults. Only
`min_samples_leaf` is tuned: at 1 a leaf can be a single household-day, at 50 the trees smooth
over real between-household differences. Sweep {1, 5, 20, 50} on the last 15% of train dates.

Picking the numerical minimum is not enough if two settings land within noise of each other.
The cell reports the standard error of each validation MAE and a paired test against the best,
then applies the one-standard-error rule: among settings within one SE of the best, take the
most constrained.

In [ ]:
# STEP 5 - HYPERPARAMETER SELECTION ON A TIME-BASED VALIDATION SLICE
val_dates = np.sort(train["date"].unique())
val_cutoff = val_dates[int(len(val_dates) * 0.85)]
m_fit = (train["date"] <= val_cutoff).values
m_val = ((train["date"] > val_cutoff) & eval_train).values

print(f"Internal validation cutoff : {pd.Timestamp(val_cutoff).date()}")
print(f"Fit rows  : {m_fit.sum():,}   Validation rows : {m_val.sum():,}\n")

leaf_grid = [1, 5, 20, 50]
leaf_scores, leaf_abs_err = [], []
for leaf in leaf_grid:
    rf_probe = RandomForestRegressor(n_estimators=200, min_samples_leaf=leaf, max_features="sqrt",
                                     n_jobs=-1, random_state=42)
    rf_probe.fit(X_train[m_fit], y_train[m_fit])
    abs_err = np.abs(y_train[m_val].values - rf_probe.predict(X_train[m_val]))
    leaf_scores.append(abs_err.mean()); leaf_abs_err.append(abs_err)
    print(f"  min_samples_leaf = {leaf:>3}   validation MAE = {abs_err.mean():.3f} kWh")

from scipy.stats import norm
_best = int(np.argmin(leaf_scores))
_ae_best = leaf_abs_err[_best]
_se_best = _ae_best.std(ddof=1) / np.sqrt(len(_ae_best))
_thresh = leaf_scores[_best] + _se_best
print(f"\nbest = leaf {leaf_grid[_best]} at {leaf_scores[_best]:.4f} kWh, SE {_se_best:.4f}, "
      f"one-SE threshold {_thresh:.4f}\n")
print(f"{'leaf':>5} {'val MAE':>9} {'SE':>7} {'within 1 SE':>12} {'diff vs best':>13} {'p':>9}")
_within = []
for i, leaf in enumerate(leaf_grid):
    d = leaf_abs_err[i] - _ae_best
    s = d.std(ddof=1) / np.sqrt(len(d))
    stat = d.mean() / s if s > 0 else np.nan
    p = 2 * norm.sf(abs(stat)) if s > 0 else np.nan
    ok = leaf_scores[i] <= _thresh
    if ok:
        _within.append(leaf)
    se_i = leaf_abs_err[i].std(ddof=1) / np.sqrt(len(leaf_abs_err[i]))
    print(f"{leaf:5d} {leaf_scores[i]:9.4f} {se_i:7.4f} {('yes' if ok else 'no'):>12} {d.mean():+13.4f} {p:9.4g}")

BEST_LEAF = max(_within)
print(f"\nWithin one SE of the best: {_within}")
print(f"One-SE rule selects the most constrained of these -> min_samples_leaf = {BEST_LEAF}")

In [ ]:
# STEP 6 - TRAIN FINAL RANDOM FOREST (full train set) + EVALUATE ON TEST
rf = RandomForestRegressor(n_estimators=300, max_features="sqrt", min_samples_leaf=BEST_LEAF,
                           n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test[eval_test.values])

print("\nRandom Forest:")
results.append(compute_metrics(y_pred_rf, "Random Forest"))

print("\n" + "=" * 80); print("SUMMARY TABLE"); print("=" * 80)
results_df = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
display(results_df.round(3))

imp = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print("\nFeature importances (mean decrease in impurity):")
display(imp.round(4).to_frame("importance"))
AR_SHARE = float(imp[["recv_lag1", "recv_lag7", "recv_roll7_mean", "recv_roll7_std"]].sum())
HDD_SHARE = float(imp["hdd_15"])
SUN_SHARE = float(imp[["Sunshine_duration_hourly_sum", "sunshine_missing"]].sum())
print(f"\nautoregressive {AR_SHARE:.1%} | hdd_15 {HDD_SHARE:.1%} | solar {SUN_SHARE:.1%} | "
      f"rest {1-AR_SHARE-HDD_SHARE-SUN_SHARE:.1%}")

## STEP 6b - Skill scores and error-tail ratio

Two derived columns on top of the metrics table:

- **skill** = 1 - error_model / error_naive1, for MAE and RMSE. Expresses accuracy as the share
  of naive error removed, which keeps later levels comparable across seasons and household mixes.
- **RMSE / MAE**. At a fixed MAE this grows with the weight of the error tail, so it separates
  tail-prone models from steady ones.

In [ ]:
# STEP 6b - SKILL SCORES vs THE NAIVE REFERENCE + ERROR-TAIL RATIO
REF_MODEL = "Naive lag-1 (yesterday)"
_ref = results_df.set_index("Model").loc[REF_MODEL]
results_df["MAE skill"] = 1 - results_df["MAE"] / _ref["MAE"]
results_df["RMSE skill"] = 1 - results_df["RMSE"] / _ref["RMSE"]
results_df["RMSE/MAE"] = results_df["RMSE"] / results_df["MAE"]

print("=" * 96); print(f"SUMMARY TABLE with skill scores (reference: {REF_MODEL})"); print("=" * 96)
display(results_df.assign(**{"MAE skill": (results_df["MAE skill"] * 100).round(1),
                             "RMSE skill": (results_df["RMSE skill"] * 100).round(1)}).round(3))
print("Skill columns are percentages of the naive lag-1 error removed; negative is worse than doing nothing.\n")

_rf = results_df.set_index("Model").loc["Random Forest"]
_sig = results_df.set_index("Model").loc["Energy signature (HDD regression)"]
print(f"Random Forest removes {_rf['MAE skill']:.1%} of the naive MAE but {_rf['RMSE skill']:.1%} of the naive RMSE.")
if _rf["RMSE skill"] > _rf["MAE skill"]:
    print("  -> more squared error removed than absolute error: the gain sits in the high-error days.")
print(f"\nError-tail ratio (RMSE/MAE): Random Forest {_rf['RMSE/MAE']:.2f}, "
      f"naive lag-1 {_ref['RMSE']/_ref['MAE']:.2f}, energy signature {_sig['RMSE/MAE']:.2f}.")

## STEP 6c - Does the training loss match the reported metric?

MAE is the reported metric, but `RandomForestRegressor` splits on squared error and predicts the
leaf mean. The mean is optimal under squared error; the median is optimal under absolute error.
Worth checking whether the mismatch costs anything.

`criterion="absolute_error"` is far more expensive than the default, so this runs on a reduced
ensemble and is off by default.

In [ ]:
# STEP 6c - MAE-CRITERION CONTROL EXPERIMENT (slow; gated)
RUN_MAE_CRITERION = False   # set True to run; expect minutes, not seconds
N_TREES_CRITERION = 100

MAE_CRIT_RESULT = None
if RUN_MAE_CRITERION:
    rf_l2 = RandomForestRegressor(n_estimators=N_TREES_CRITERION, max_features="sqrt",
                                  min_samples_leaf=BEST_LEAF, n_jobs=-1, random_state=42).fit(X_train, y_train)
    rf_l1 = RandomForestRegressor(n_estimators=N_TREES_CRITERION, max_features="sqrt",
                                  min_samples_leaf=BEST_LEAF, criterion="absolute_error",
                                  n_jobs=-1, random_state=42).fit(X_train, y_train)
    mae_l2 = mean_absolute_error(y_eval, rf_l2.predict(X_test[eval_test.values]))
    mae_l1 = mean_absolute_error(y_eval, rf_l1.predict(X_test[eval_test.values]))
    MAE_CRIT_RESULT = {"n_estimators": N_TREES_CRITERION, "squared_error": mae_l2,
                       "absolute_error": mae_l1, "delta": mae_l2 - mae_l1}
    print(f"Matched {N_TREES_CRITERION}-tree comparison, test MAE:")
    print(f"  criterion='squared_error'  (leaf mean)   : {mae_l2:.3f} kWh")
    print(f"  criterion='absolute_error' (leaf median) : {mae_l1:.3f} kWh")
    print(f"  difference                                : {mae_l2 - mae_l1:+.3f} kWh")
else:
    print("Skipped (RUN_MAE_CRITERION = False).")

In [ ]:
# STEP 7 - EVALUATION PLOTS
_res_rf = y_eval - y_pred_rf
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(y_eval, y_pred_rf, s=4, alpha=0.2)
_lim = [0, np.percentile(y_eval, 99.5)]
axes[0].plot(_lim, _lim, color="black", lw=1)
axes[0].set_xlim(_lim); axes[0].set_ylim(_lim)
axes[0].set_xlabel("actual (kWh)"); axes[0].set_ylabel("predicted (kWh)")
axes[0].set_title("Random Forest: predicted vs actual")
axes[1].hist(_res_rf, bins=60)
axes[1].axvline(0, color="black", lw=1)
axes[1].set_xlabel("residual (actual - predicted, kWh)"); axes[1].set_title("Residual distribution")
imp.head(10)[::-1].plot.barh(ax=axes[2])
axes[2].set_title("Top 10 feature importances")
plt.tight_layout(); plt.show()

## STEP 7b - Per-household error distribution

The pooled MAE hides the spread in finding 2 and implicitly weights large consumers more. The
fleet view below shows how accuracy is distributed, which is what decides whether one pooled
model is acceptable.

In [ ]:
# STEP 7b - PER-HOUSEHOLD TEST MAE DISTRIBUTION
_per_hh = test.loc[eval_test, ["Household_ID"]].copy()
_per_hh["abs_rf"] = np.abs(y_eval - y_pred_rf)
_per_hh["abs_lag1"] = np.abs(y_eval - test.loc[eval_test, "recv_lag1"].values)
_per_hh["abs_sig"] = np.abs(y_eval - y_pred_sig)
hh_mae = _per_hh.groupby("Household_ID", observed=True).mean()
hh_mae.columns = ["Random Forest", "Naive lag-1", "Energy signature"]
hh_mae = hh_mae[["Naive lag-1", "Energy signature", "Random Forest"]]

hh_summary = (hh_mae.describe(percentiles=[0.25, 0.5, 0.75]).T[["25%", "50%", "75%", "max"]]
              .rename(columns={"25%": "p25", "50%": "median", "75%": "p75", "max": "worst"}).round(2))
print("Per-household test MAE distribution (kWh):")
display(hh_summary)
_share_rf_better = (hh_mae["Random Forest"] < hh_mae["Naive lag-1"]).mean()
print(f"Households where RF beats naive lag-1 : {_share_rf_better:.1%} of {len(hh_mae)}")

fig, ax = plt.subplots(figsize=(8, 4))
hh_mae.boxplot(ax=ax, grid=False)
ax.set_ylabel("Per-household test MAE (kWh)")
ax.set_title("Distribution of per-household accuracy across the test fleet")
plt.tight_layout(); plt.show()
print("\nFive hardest households for the RF (test MAE, kWh):")
display(hh_mae.sort_values("Random Forest", ascending=False).head(5).round(2))

## STEP 7c - Residuals against temperature

If the forest has absorbed the heating signal, its signed residuals should show no trend across
the temperature range. Bin test rows by `hdd_15` and compare the mean signed residual per bin. A
model that missed part of the heating response under-predicts cold days, i.e. positive residuals
at high HDD.

In [ ]:
# STEP 7c - MEAN SIGNED RESIDUAL BY hdd_15 BIN
_res = test.loc[eval_test, ["hdd_15"]].copy()
_res["Naive lag-1"] = y_eval - test.loc[eval_test, "recv_lag1"].values
_res["Energy signature"] = y_eval - y_pred_sig
_res["Random Forest"] = _res_rf
_res["hdd_bin"] = pd.cut(_res["hdd_15"], bins=8)
resid_bins = _res.groupby("hdd_bin", observed=True)[["Naive lag-1", "Energy signature", "Random Forest"]].mean().round(3)
resid_bins["n_rows"] = _res.groupby("hdd_bin", observed=True).size()
print("Mean signed residual (kWh) by hdd_15 bin:")
display(resid_bins)

fig, ax = plt.subplots(figsize=(9, 4))
_x = np.arange(len(resid_bins))
for col, marker in [("Naive lag-1", "o"), ("Energy signature", "s"), ("Random Forest", "D")]:
    ax.plot(_x, resid_bins[col], marker=marker, label=col)
ax.axhline(0, color="black", lw=1)
ax.set_xticks(_x); ax.set_xticklabels([str(i) for i in resid_bins.index], rotation=30, ha="right", fontsize=8)
ax.set_xlabel("hdd_15 bin (cold ->)"); ax.set_ylabel("Mean signed residual (kWh)")
ax.set_title("Temperature-conditional bias")
ax.legend(); plt.tight_layout(); plt.show()

MAX_BIAS_RF = resid_bins["Random Forest"].abs().max()
MAX_BIAS_SIG = resid_bins["Energy signature"].abs().max()
print(f"Largest absolute bin bias : RF {MAX_BIAS_RF:.2f} kWh vs energy signature {MAX_BIAS_SIG:.2f} kWh")

## STEP 7d - Per-household diagnostics and files for Level 1

Everything above computes predictions, prints a number, and discards the predictions. Write them
out instead, so Level 1 starts from measurements rather than from scratch.

Two files: the test predictions per household-day, and one diagnostic row per household. Four
columns in the second are meant to be used, not just read.

- `mase_rf` - error scaled by that household's own naive baseline, comparable across the fleet
- `bias_rf` - mean signed residual; positive means we under-forecast
- `tail_ratio` - RMSE / MAE within the household
- `hdd_slope` - slope of the residual against `hdd_15`. Non-zero means the pooled model has that
  household's heating response wrong, which an integer household encoding cannot fix.

In [ ]:
# STEP 7d - PER-HOUSEHOLD DIAGNOSTICS + FILES FOR LEVEL 1
pred_test = test.loc[eval_test, ["Household_ID", "date", TARGET, "hdd_15",
                                 "Sunshine_duration_hourly_sum", "Installation_HasPVSystem"]].copy()
pred_test["pred_rf"] = y_pred_rf
pred_test["pred_naive1"] = test.loc[eval_test, "recv_lag1"].values
pred_test["pred_signature"] = y_pred_sig
pred_test["resid_rf"] = pred_test[TARGET].values - pred_test["pred_rf"].values
pred_test["mase_scale"] = mase_scale_test
PRED_PATH = PROCESSED / "level0_test_predictions.parquet"
pred_test.to_parquet(PRED_PATH, index=False)
print(f"Saved {len(pred_test):,} test predictions -> {PRED_PATH.relative_to(REPO_ROOT)}")

_rows = []
for hh, g in pred_test.groupby("Household_ID", observed=True, sort=True):
    e = g["resid_rf"].values
    abs_e = np.abs(e)
    x = g["hdd_15"].values
    ok = np.isfinite(x) & np.isfinite(e)
    slope = np.polyfit(x[ok], e[ok], 1)[0] if (ok.sum() >= 10 and np.std(x[ok]) > 1e-6) else np.nan
    mae_rf = abs_e.mean()
    _rows.append({"Household_ID": hh, "n_days": len(g), "mae_rf": mae_rf,
                  "rmse_rf": float(np.sqrt((e ** 2).mean())),
                  "mae_naive1": float(np.abs(g[TARGET].values - g["pred_naive1"].values).mean()),
                  "mae_signature": float(np.abs(g[TARGET].values - g["pred_signature"].values).mean()),
                  "mase_rf": float((abs_e / g["mase_scale"].values).mean()),
                  "bias_rf": float(e.mean()),
                  "tail_ratio": float(np.sqrt((e ** 2).mean()) / mae_rf) if mae_rf > 1e-9 else np.nan,
                  "hdd_slope": slope})
hh_diag = pd.DataFrame(_rows)
hh_diag["rf_beats_naive"] = hh_diag["mae_rf"] < hh_diag["mae_naive1"]
DIAG_PATH = REPORTS_DIR / "level0_per_household_errors.csv"
hh_diag.to_csv(DIAG_PATH, index=False)
print(f"Saved {len(hh_diag)} per-household diagnostic rows -> {DIAG_PATH.relative_to(REPO_ROOT)}\n")

N_HH_TEST = len(hh_diag)
N_HH_RF_LOSES = int((~hh_diag["rf_beats_naive"]).sum())
SHARE_RF_WINS = float(hh_diag["rf_beats_naive"].mean())
WORST_HH_MAE = hh_diag["mae_rf"].max()
MEDIAN_HH_MAE = hh_diag["mae_rf"].median()
SHARE_UNDER = float((hh_diag["bias_rf"] > 0).mean())
SHARE_SLOPE = float((hh_diag["hdd_slope"].abs() > 0.1).mean())
display(hh_diag[["n_days", "mae_rf", "mase_rf", "bias_rf", "tail_ratio", "hdd_slope"]]
        .describe(percentiles=[0.25, 0.5, 0.75]).round(3))
print(f"RF beats naive lag-1 in {SHARE_RF_WINS:.1%} of {N_HH_TEST} test households, so it loses in {N_HH_RF_LOSES}.")
print(f"Worst-served household MAE {WORST_HH_MAE:.2f} kWh vs a median of {MEDIAN_HH_MAE:.2f} kWh.")
print(f"{SHARE_UNDER:.1%} of households are under-forecast on average (positive bias).")
print(f"{SHARE_SLOPE:.1%} of households have |residual slope on hdd_15| > 0.1 kWh/degree-day.")
print("\nHouseholds where the pooled forest LOSES to naive persistence:")
display(hh_diag[~hh_diag["rf_beats_naive"]][["Household_ID", "n_days", "mae_rf", "mae_naive1",
                                             "bias_rf", "hdd_slope"]].round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(hh_diag["hdd_slope"], hh_diag["mase_rf"], alpha=0.7)
axes[0].axvline(0, color="black", lw=1); axes[0].axhline(1, color="crimson", lw=1, ls="--")
axes[0].set_xlabel("residual slope on hdd_15 (kWh per degree-day)"); axes[0].set_ylabel("per-household MASE")
axes[0].set_title("Residual slope on hdd_15 vs MASE (>1 = worse than persistence)")
axes[1].scatter(hh_diag["bias_rf"], hh_diag["tail_ratio"], alpha=0.7)
axes[1].axvline(0, color="black", lw=1)
axes[1].set_xlabel("mean signed residual (kWh; + = under-forecast)"); axes[1].set_ylabel("RMSE / MAE within household")
axes[1].set_title("Bias vs error-tail weight")
plt.tight_layout(); plt.show()

In [ ]:
# STEP 8 - TIME SERIES COMPARISON (example household)
example_hh = pred_test["Household_ID"].value_counts().index[0]
_e = pred_test[pred_test["Household_ID"] == example_hh].sort_values("date")
hh_mae_rf = float(np.abs(_e[TARGET] - _e["pred_rf"]).mean())
hh_mae_base = float(np.abs(_e[TARGET] - _e["pred_naive1"]).mean())
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(_e["date"], _e[TARGET], label="actual", lw=1.4)
ax.plot(_e["date"], _e["pred_rf"], label="Random Forest", lw=1.2)
ax.plot(_e["date"], _e["pred_naive1"], label="naive lag-1", lw=0.9, alpha=0.7)
ax.set_ylabel("gross load (kWh)")
ax.set_title(f"Household {example_hh}: RF MAE {hh_mae_rf:.2f} vs naive lag-1 {hh_mae_base:.2f} kWh")
ax.legend(); plt.tight_layout(); plt.show()
print(f"Household {example_hh}:  RF MAE = {hh_mae_rf:.3f} kWh  |  Naive t-1 MAE = {hh_mae_base:.3f} kWh")

## STEP 8b - Rolling-origin check

The single split stays the primary protocol. Its weakness is that every headline number rests on
one eight-month window. Here the comparison is re-run from eight month-end origins, fitting on
everything up to the origin and evaluating on the following month.

- Hyperparameters frozen at the STEP 5 value. No per-fold re-tuning.
- Folds after the first include former test months in their training data. That is inherent to
  rolling origin and affects only this table.
- MAE per fold. The question is whether the ranking is stable.

In [ ]:
# STEP 8b - ROLLING-ORIGIN EVALUATION (monthly origins, frozen hyperparameters)
full = pd.concat([train, test], ignore_index=True)
full_eval = full[EVAL_COLS].notna().all(axis=1)

_origin_strs = ["2023-06-30", "2023-07-31", "2023-08-31", "2023-09-30",
                "2023-10-31", "2023-11-30", "2023-12-31", "2024-01-31"]
_end_strs = _origin_strs[1:] + ["2024-03-01"]
_origins = [pd.Timestamp(s) for s in _origin_strs]
_ends = [pd.Timestamp(s) for s in _end_strs]

def _prep(df):
    X = df[FEATURE_COLS].copy()
    for col in FEATURE_COLS:
        if str(X[col].dtype) in ("category", "bool", "boolean"):
            X[col] = X[col].astype("float64") if str(X[col].dtype) == "boolean" \
                else X[col].astype("category").cat.codes
    return X

roll_rows = []
for k, (o, e) in enumerate(zip(_origins, _ends), start=1):
    m_tr = full["date"] <= o
    m_te = (full["date"] > o) & (full["date"] <= e) & full_eval
    f_tr, f_te = full[m_tr], full[m_te]

    enc = {hh: i for i, hh in enumerate(sorted(f_tr["Household_ID"].unique()))}
    f_tr = f_tr.assign(household_enc=f_tr["Household_ID"].map(enc).fillna(-1).astype(int))
    f_te = f_te.assign(household_enc=f_te["Household_ID"].map(enc).fillna(-1).astype(int))

    mae_naive = mean_absolute_error(f_te[TARGET], f_te["recv_lag1"])

    a_k, b_k = {}, {}
    for hh, gg in f_tr.groupby("Household_ID", observed=True):
        gg = gg[["hdd_15", TARGET]].dropna()
        if len(gg) >= MIN_TRAIN_DAYS and gg["hdd_15"].std() > 1e-6:
            s, i0 = np.polyfit(gg["hdd_15"].values, gg[TARGET].values, 1)
            a_k[hh], b_k[hh] = i0, s
    _t = f_tr[["hdd_15", TARGET]].dropna()
    ps, pi = np.polyfit(_t["hdd_15"].values, _t[TARGET].values, 1)
    a_v = f_te["Household_ID"].map(a_k).fillna(pi).values
    b_v = f_te["Household_ID"].map(b_k).fillna(ps).values
    hdd_v = f_te["hdd_15"].fillna(f_te["hdd_15"].median()).values
    mae_sig = mean_absolute_error(f_te[TARGET], np.clip(a_v + b_v * hdd_v, 0.0, None))

    rf_k = RandomForestRegressor(n_estimators=300, max_features="sqrt", min_samples_leaf=BEST_LEAF,
                                 n_jobs=-1, random_state=42).fit(_prep(f_tr), f_tr[TARGET])
    mae_rf = mean_absolute_error(f_te[TARGET], rf_k.predict(_prep(f_te)))

    roll_rows.append({"fold": k, "origin": str(o.date()), "horizon_end": str(e.date()),
                      "n_test_rows": int(m_te.sum()), "MAE naive lag-1": mae_naive,
                      "MAE energy signature": mae_sig, "MAE Random Forest": mae_rf})
    print(f"fold {k}: origin {o.date()}  ->  naive {mae_naive:.3f} | signature {mae_sig:.3f} | RF {mae_rf:.3f}")

roll_df = pd.DataFrame(roll_rows)
_mcols = ["MAE naive lag-1", "MAE energy signature", "MAE Random Forest"]
roll_summary = roll_df[_mcols].agg(["mean", "std"]).round(3)
print("\nRolling-origin summary (kWh):"); display(roll_summary)
_rf_wins = int((roll_df["MAE Random Forest"] < roll_df[_mcols[:2]].min(axis=1)).sum())
print(f"RF is the most accurate model in {_rf_wins} of {len(roll_df)} folds")

## STEP 8c - Is the win real?

The forest is ahead in every fold, but fold-to-fold MAE swings with the season, so a difference
of means proves nothing on its own. The comparison is paired (every model sees the same fold),
so the quantity to test is the per-fold difference.

Two tests: a sign test on folds won, and Diebold-Mariano on the paired per-observation loss
differences. The second is the one Level 1 needs, because its margins will be much smaller.

In [ ]:
# STEP 8c - SIGNIFICANCE OF THE ROLLING-ORIGIN RESULT
from scipy.stats import binomtest

_diff = roll_df["MAE naive lag-1"] - roll_df["MAE Random Forest"]
N_FOLDS = len(_diff)
N_RF_WINS = int((_diff > 0).sum())
SIGN_P = float(binomtest(N_RF_WINS, n=N_FOLDS, alternative="greater").pvalue)
GAIN_MEAN = float(_diff.mean()); GAIN_SD = float(_diff.std(ddof=1))
print("Paired per-fold comparison: Random Forest vs naive lag-1")
print("-" * 62)
print(f"  folds won by RF          : {N_RF_WINS} of {N_FOLDS}")
print(f"  one-sided sign-test p    : {SIGN_P:.4f}")
print(f"  mean per-fold MAE gain   : {GAIN_MEAN:+.3f} kWh (SD of the paired differences {GAIN_SD:.3f})")
display(roll_df[["fold", "origin", "MAE naive lag-1", "MAE Random Forest"]]
        .assign(**{"gain (kWh)": _diff.round(3), "RF wins": _diff > 0}).round(3))

def diebold_mariano(e1, e2, power=1, h=1):
    """One-sided DM statistic. H0: equal accuracy. HA: model 2 is more accurate.

    e1, e2: error vectors on the same observations. power=1 -> absolute-error loss
    (matches MAE), power=2 -> squared-error loss. h>1 adds Newey-West autocovariance lags.
    """
    e1, e2 = np.asarray(e1, float), np.asarray(e2, float)
    d = np.abs(e1) ** power - np.abs(e2) ** power
    n = len(d); d_bar = d.mean()
    gamma0 = np.mean((d - d_bar) ** 2)
    gamma = [np.mean((d[k:] - d_bar) * (d[:-k] - d_bar)) for k in range(1, h)]
    var_d = (gamma0 + 2 * sum(gamma)) / n
    stat = d_bar / np.sqrt(var_d) if var_d > 0 else np.nan
    from scipy.stats import norm
    # norm.sf is the upper-tail probability; 1 - norm.cdf underflows to 0 for large statistics.
    return stat, float(norm.sf(stat))

DM_STAT, DM_P = diebold_mariano(y_eval - test.loc[eval_test, "recv_lag1"].values, y_eval - y_pred_rf, power=1)
print(f"\nDiebold-Mariano on the single-split test set (absolute-error loss, h=1):")
print(f"  DM statistic = {DM_STAT:.2f}, one-sided p = {DM_P:.3g}")
print("\nLevel 1 reuses this function on its own paired errors.")

## STEP 9 - Write the report

Feature-selection decisions, hyperparameter choice and the metric tables go to
`reports/level0_baseline_report.md`.

In [ ]:
# STEP 9 - WRITE OWNED NARRATIVE REPORT
from datetime import datetime, timezone

_r = results_df.set_index("Model")
_mae_rf = _r.loc["Random Forest", "MAE"]
_mae_lag1 = _r.loc["Naive lag-1 (yesterday)", "MAE"]
_mae_sig = _r.loc["Energy signature (HDD regression)", "MAE"]

lines = [
    "# LEVEL 0 BASELINE REPORT", "",
    f"Generated by `notebooks/modelling/07_level0_baseline.ipynb` on {datetime.now(timezone.utc).isoformat()}",
    "",
    "**Scope:** Level 0 baseline ladder - three naive forecasts, a per-household degree-day",
    "energy-signature regression, and a single pooled Random Forest predicting `gross_load` one",
    "day ahead, on the stage-3 panel.", "",
    f"**Weather:** every weather column is the d-1 observation (`weather_shift_days = {WEATHER_SHIFT}`).",
    "No target-day weather enters the model, so accuracy is not inflated by a perfect-forecast",
    "assumption.", "",
    "## 0. Data-understanding findings (rechecked on the train panel)", "",
    f"- corr(daily portfolio median load, median temperature d-1) r = {R_TEMP:+.3f}; with median `hdd_15` r = {R_HDD:+.3f}.",
    f"- Per-household median daily load: min {hh_median_load.min():.2f}, p10 {hh_median_load.quantile(.10):.2f}, "
    f"median {hh_median_load.median():.2f}, p90 {hh_median_load.quantile(.90):.2f}, max {hh_median_load.max():.2f} kWh. "
    f"Max/min is {SPREAD_RATIO:.0f}x but both extremes are short-history households; "
    f"p90/p10 = {hh_median_load.quantile(.90)/hh_median_load.quantile(.10):.1f}x.",
    f"- {PV_TRUE} PV vs {PV_FALSE} non-PV train households; median per-household CV {CV_PV:.2f} (PV) vs {CV_NONPV:.2f} (non-PV).",
    "",
    "### 0b. Temperature dependence at household resolution", "",
    f"Across the {HH_N_CORR} train households with at least 30 days: median corr(`gross_load`, `hdd_15`) "
    f"= {HH_R_MEDIAN:+.3f} (IQR {HH_R_P25:+.3f} to {HH_R_P75:+.3f}, range {HH_R_MIN:+.3f} to {HH_R_MAX:+.3f}). "
    f"Explained variance falls from {R_HDD ** 2:.0%} at portfolio level to a median of {HH_R2_MEDIAN:.0%} per household.",
    "",
    "## 1. Feature selection", "",
    f"Candidate pool: the {len(SCHEMA_FEATURES)} active features from `model_table_schema.json`, plus "
    "`household_enc` and `sunshine_missing` added at the modelling stage. After collinearity pruning "
    f"the frozen set has {len(FEATURE_COLS)} features:", "",
    "| feature |", "| --- |",
] + [f"| `{c}` |" for c in FEATURE_COLS] + [
    "",
    f"Dropped: {', '.join('`'+c+'`' for c in sorted(DROPPED))}.",
    "",
    "## 2. Evaluation protocol and metrics", "",
    f"Split owned by stage 3 (cutoff {SCHEMA['split_cutoff']}). Train {len(train):,} rows / "
    f"{train['Household_ID'].nunique()} households; test {len(test):,} rows / {test['Household_ID'].nunique()} households.",
    "",
    f"All models are scored on the {N_EVAL_TEST:,} test rows ({eval_test.mean():.1%}) where every naive "
    "forecast is defined, so the comparison is like for like. Metrics: MAE (primary), RMSE, MASE "
    f"(fleet median scale {FLEET_Q:.3f} kWh; {N_FALLBACK_ROWS} rows on the fallback). R2 is not reported.",
    "",
    "## 3. Hyperparameter selection", "",
    f"`min_samples_leaf` swept over {leaf_grid} on a time-based validation slice (train dates after "
    f"{pd.Timestamp(val_cutoff).date()}, {int(m_val.sum()):,} rows); validation MAE "
    f"{', '.join(f'{v:.3f}' for v in leaf_scores)} kWh.",
    "",
    f"Settings within one standard error of the best: {_within} (SE {_se_best:.4f} kWh). The "
    f"one-standard-error rule selects the most constrained of these, hence **min_samples_leaf = {BEST_LEAF}**.",
    "",
    "## 4. Energy-signature benchmark", "",
    f"Per-household OLS of `gross_load` on `hdd_15`: {N_SIG_HH} households fitted individually "
    f"(>= {MIN_TRAIN_DAYS} train days). Median signature: base load {np.median(list(sig_a.values())):.2f} kWh, "
    f"heating response {np.median(list(sig_b.values())):.2f} kWh per degree-day.",
    "",
    "## 5. Test-set results (single split)", "",
    df_to_md(results_df.round(3)), "",
    f"Random Forest improves test MAE by {(1 - _mae_rf / _mae_lag1):.1%} over naive lag-1 and by "
    f"{(1 - _mae_rf / _mae_sig):.1%} over the temperature-only energy signature.",
    "",
    f"Feature importances: autoregressive {AR_SHARE:.1%}, `hdd_15` {HDD_SHARE:.1%}, "
    f"solar (sunshine + missingness indicator) {SUN_SHARE:.1%}, remaining features "
    f"{1-AR_SHARE-HDD_SHARE-SUN_SHARE:.1%}.",
    "",
    "## 6. Per-household error distribution (test)", "",
    df_to_md(hh_summary.reset_index().rename(columns={"index": "model"})), "",
    f"RF beats naive lag-1 in {SHARE_RF_WINS:.1%} of the {N_HH_TEST} test households and loses in {N_HH_RF_LOSES}.",
    "",
    df_to_md(hh_diag[["mae_rf", "mase_rf", "bias_rf", "tail_ratio", "hdd_slope"]]
             .describe(percentiles=[0.25, 0.5, 0.75]).T[["25%", "50%", "75%", "max"]]
             .rename(columns={"25%": "p25", "50%": "median", "75%": "p75", "max": "max"})
             .round(3).reset_index().rename(columns={"index": "metric"})), "",
    f"Worst-served household MAE {WORST_HH_MAE:.2f} kWh against a median of {MEDIAN_HH_MAE:.2f} kWh. "
    f"{SHARE_UNDER:.1%} of households are under-forecast on average; {SHARE_SLOPE:.1%} carry a residual "
    "slope on `hdd_15` above 0.1 kWh per degree-day in absolute value.",
    "",
    "Artefacts for the later levels:", "",
    f"- `{PRED_PATH.relative_to(REPO_ROOT)}`", f"- `{DIAG_PATH.relative_to(REPO_ROOT)}`",
    f"- `{SIG_PATH.relative_to(REPO_ROOT)}`", "",
    "## 7. Temperature-conditional bias (test)", "",
    df_to_md(resid_bins.reset_index().astype(str)), "",
    f"Largest absolute bin bias: Random Forest {MAX_BIAS_RF:.2f} kWh vs energy signature {MAX_BIAS_SIG:.2f} kWh.",
    "",
    "## 8. Rolling-origin robustness", "",
    df_to_md(roll_df.round(3)), "", df_to_md(roll_summary.reset_index().rename(columns={"index": "stat"})), "",
    f"The Random Forest is the most accurate model in {_rf_wins} of {len(roll_df)} folds.",
    "",
    "### 8b. Significance", "",
    f"Paired per-fold comparison against naive lag-1: {N_RF_WINS} of {N_FOLDS} folds won, one-sided "
    f"sign-test p = {SIGN_P:.4f}; mean per-fold MAE gain {GAIN_MEAN:+.3f} kWh (SD of the paired "
    f"differences {GAIN_SD:.3f} kWh).",
    "",
    f"Diebold-Mariano on the single split (absolute-error loss, h = 1): DM = {DM_STAT:.2f}, "
    f"one-sided p = {DM_P:.3g}.",
    "",
    "## 9. Loss-versus-metric check", "",
    (f"Matched {MAE_CRIT_RESULT['n_estimators']}-tree comparison, test MAE: squared_error "
     f"{MAE_CRIT_RESULT['squared_error']:.3f} vs absolute_error {MAE_CRIT_RESULT['absolute_error']:.3f} kWh "
     f"(difference {MAE_CRIT_RESULT['delta']:+.3f})."
     if MAE_CRIT_RESULT else
     "Not run in this execution (`RUN_MAE_CRITERION = False`). The forest splits on squared error and "
     "predicts the leaf mean, which is RMSE-optimal, while MAE is the primary metric and is minimised "
     "by the conditional median."),
    "",
    f"Example household {example_hh}: RF MAE = {hh_mae_rf:.3f} kWh vs naive lag-1 MAE = {hh_mae_base:.3f} kWh.",
    "",
]
report_path = REPORTS_DIR / "level0_baseline_report.md"
report_path.write_text("\n".join(lines))
print(f"Saved -> {report_path.relative_to(REPO_ROOT)}")